# 🗡️ Hey Aragorn Wake Word Training

Train a custom wake word model for "Hey Aragorn" using micro-wake-word.

**Run each cell in order.**

## Step 0: Free Disk Space

Removes large pre-installed system components that this notebook never uses. Run this first before anything else.

In [ ]:
import shutil, os

print("Disk before cleanup:")
!df -h / | tail -1
print()

# Docs and man pages — never needed in a notebook environment
shutil.rmtree('/usr/share/doc', ignore_errors=True)
shutil.rmtree('/usr/share/man', ignore_errors=True)
shutil.rmtree('/usr/share/locale', ignore_errors=True)
print("Removed: /usr/share/doc, man, locale")

# Google Cloud SDK — ~500 MB, not used here
shutil.rmtree('/usr/lib/google-cloud-sdk', ignore_errors=True)
print("Removed: Google Cloud SDK")

# Android SDK if present — can be ~1 GB
shutil.rmtree('/usr/local/android-sdk', ignore_errors=True)
shutil.rmtree('/usr/local/julia-1.9.4', ignore_errors=True)
print("Removed: Android SDK, Julia")

# Colab sample data
shutil.rmtree('/content/sample_data', ignore_errors=True)
print("Removed: sample_data")

# APT cache
!apt-get clean -qq
!apt-get autoremove -y -qq
print("Cleaned apt cache")

# pip cache
!pip cache purge -q
print("Cleaned pip cache")

print()
print("Disk after cleanup:")
!df -h / | tail -1


## Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Step 2: Install Dependencies

In [ ]:
print("Installing system dependency espeak-ng...")
!apt-get install -y -q espeak-ng
!apt-get install -y -q libespeak-ng-dev

print("\nInstalling piper-tts (visible output - check for errors)...")
!pip install piper-tts

print("\nVerifying piper import immediately after install...")
!python -c "from piper import PiperVoice, SynthesisConfig; print('piper import: OK')"

print("\nRemoving conflicting packages...")
!pip uninstall -y numpy scipy 2>/dev/null
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tf-keras tensorflow-text opencv-python opencv-python-headless opencv-contrib-python shap ydf grain pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/numpy/*' -delete 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/scipy/*' -delete 2>/dev/null

print("\nInstalling pinned versions...")
!pip install --force-reinstall --no-cache-dir numpy==1.26.4 scipy==1.13.1
!pip install --quiet tensorflow==2.16.2 protobuf==4.25.3 ml-dtypes==0.3.2 2>/dev/null
!pip install --quiet onnxruntime 2>/dev/null
!pip install --quiet pyyaml datasets mmap-ninja tqdm audiomentations webrtcvad-wheels huggingface_hub 2>/dev/null

print("\n\u2705 Done! Restart the runtime now (Runtime -> Restart session). Then continue from Step 3.")


## ⚠️ Restart Required

**Restart the runtime before continuing!**

Click: **Runtime → Restart session** (Ctrl+M .)

Then run from **Step 3** downward.

## Step 3: Verify Install & Clone Repositories

In [ ]:
import subprocess, os, sys

# Verify numpy/scipy
import numpy as np
assert np.__version__ == '1.26.4', f'Bad numpy: {np.__version__}'
print(f'  numpy {np.__version__}: OK')
import scipy
from scipy.signal import resample
from scipy.io import wavfile
print(f'  scipy {scipy.__version__}: OK')
import tensorflow as tf
print(f'  tensorflow {tf.__version__}: OK')

# Verify piper survived the restart + numpy re-pin
smoke_cmd = [sys.executable, '-c', 'from piper import PiperVoice, SynthesisConfig; print("piper: OK")']
r = subprocess.run(smoke_cmd, capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout.strip())
else:
    print("piper import FAILED:")
    print(r.stderr)
    print("\nTrying re-install of piper-tts...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'piper-tts'], check=True)

# Clone microWakeWord and install with its dependencies
if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', 'https://github.com/kahrendt/microWakeWord.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'microWakeWord'], check=True)
print('microWakeWord: OK')

# Clone piper-sample-generator (just clone — piper-tts already installed above)
if not os.path.exists('piper-sample-generator'):
    subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper-sample-generator.git'], check=True)
print('piper-sample-generator: cloned')

# Re-pin numpy/scipy in case anything drifted them
repin = [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
         '--no-cache-dir', 'numpy==1.26.4', 'scipy==1.13.1']
subprocess.run(repin, check=True)
print('numpy/scipy: re-pinned')

print('\n\u2705 All set!')


## Step 4: Download Piper Voice Model

In [ ]:
import os, urllib.request

os.makedirs('piper-sample-generator/models', exist_ok=True)
url  = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'

if not os.path.exists(path):
    print('Downloading...')
    urllib.request.urlretrieve(url, path)
    print('\u2705 Done!')
else:
    print('\u2705 Already exists')


## Step 5: Configure

In [ ]:
TARGET_WORD    = 'hey_air_uh_gorn'
NUM_SAMPLES    = 1000
TRAINING_STEPS = 10000

print(f'Word: {TARGET_WORD}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Steps: {TRAINING_STEPS}')


## Step 6: Generate Test Sample

Generate 1 sample to verify phonetic spelling.

**Listen to the audio.** Adjust `TARGET_WORD` in Step 5 if needed.

Tips: underscores between syllables (`hey_air_uh_gorn`), `sh`/`ch`/`th` for those sounds, `ee`/`oo` for long vowels.

In [ ]:
import subprocess, os, sys
from IPython.display import Audio, display

os.makedirs('generated_samples', exist_ok=True)
print('Generating 1 test sample...')

# PYTHONPATH tells Python where to find the piper_sample_generator package.
# piper itself (PiperVoice etc.) is installed system-wide via piper-tts in Step 2.
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD,
       '--max-samples', '1',
       '--batch-size', '1',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    wavs = sorted([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    if wavs:
        print(f'Phonetic spelling: "{TARGET_WORD}"')
        print('Does this sound right? If not, tweak TARGET_WORD in Step 5.\n')
        display(Audio(os.path.join('generated_samples', wavs[0])))
    else:
        print('No .wav files found.')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 7: Generate All Samples

In [ ]:
import subprocess, os, sys

print(f'Generating {NUM_SAMPLES} samples...')

env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD,
       '--max-samples', str(NUM_SAMPLES),
       '--batch-size', '100',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f'\u2705 Generated {count} samples!')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 8: Download Augmentation Data

In [ ]:
import os

os.makedirs('mit_rirs', exist_ok=True)
if not os.listdir('mit_rirs'):
    print('Downloading RIRs + pointsource noises (~1.3 GB, 5-10 min)...')
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print('Extracted!')
else:
    print('Already downloaded')

noise_dir = 'mit_rirs/RIRS_NOISES/pointsource_noises'
if os.path.exists(noise_dir):
    n = len([f for f in os.listdir(noise_dir) if f.endswith('.wav')])
    print(f'\u2705 {n} background noise files ready in {noise_dir}')
else:
    print(f'\u274c {noise_dir} missing - check zip extraction')


## 🧹 Disk Cleanup (After Step 8)

Run this to free up space before generating spectrograms and downloading negative datasets.

In [ ]:
import os, subprocess, sys

print('Cleaning up to free disk space...')

# Delete the downloaded zip (already extracted, no longer needed)
if os.path.exists('/tmp/rirs_noises.zip'):
    os.remove('/tmp/rirs_noises.zip')
    print('  Deleted /tmp/rirs_noises.zip')

# Purge pip download cache
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('  pip cache purged')

!df -h /


## Step 9: Generate Spectrograms

In [ ]:
import os, sys
if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

print('Generating spectrograms...')

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['mit_rirs/RIRS_NOISES/pointsource_noises'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

os.makedirs('generated_augmented_features/training', exist_ok=True)

spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    slide_frames=10,
    step_ms=10,
)

RaggedMmap.from_generator(
    out_dir='generated_augmented_features/training/wakeword_mmap',
    sample_generator=spectrograms.spectrogram_generator(split='train', repeat=2),
    batch_size=100,
    verbose=True,
)

print('\u2705 Done!')


## Step 10: Download Negative Datasets

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download, list_repo_files

print('Discovering negative datasets on HuggingFace...')
os.makedirs('negative_datasets', exist_ok=True)

repo_id   = 'kahrendt/microwakeword'
repo_type = 'dataset'
zip_files = [f for f in list_repo_files(repo_id, repo_type=repo_type) if f.endswith('.zip')]
print(f'Found {len(zip_files)} zip files: {zip_files}')

for fname in zip_files:
    base    = os.path.splitext(os.path.basename(fname))[0]
    out_dir = f'negative_datasets/{base}'
    if not os.path.exists(out_dir):
        print(f'Downloading {fname}...')
        local = hf_hub_download(repo_id=repo_id, filename=fname, repo_type=repo_type)
        with zipfile.ZipFile(local, 'r') as zf:
            zf.extractall('negative_datasets')
        print(f'  done: {base}')
    else:
        print(f'  already exists: {base}')

neg_dirs = [d for d in os.listdir('negative_datasets') if os.path.isdir(f'negative_datasets/{d}')]
print(f'\n\u2705 Negative dataset directories: {neg_dirs}')


## 🧹 Disk Cleanup (After Step 10)

Run this to remove HuggingFace's local zip cache — the datasets are already extracted.

In [ ]:
import os, shutil, subprocess

print('Cleaning up HuggingFace download cache...')

hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(os.path.getsize(os.path.join(dp, f))
               for dp, _, files in os.walk(hf_cache) for f in files)
    shutil.rmtree(hf_cache)
    print(f'  Deleted HuggingFace cache ({size/1e9:.1f} GB freed)')
else:
    print('  No HuggingFace cache found')

!df -h /


## Step 11: Create Config

In [ ]:
import yaml, os

neg_dirs   = sorted([d for d in os.listdir('negative_datasets') if os.path.isdir(f'negative_datasets/{d}')])
eval_dirs  = [d for d in neg_dirs if 'eval' in d]
train_dirs = [d for d in neg_dirs if 'eval' not in d]
print(f'Train negatives : {train_dirs}')
print(f'Eval  negatives : {eval_dirs}')

neg_features = []
for d in train_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}', 'sampling_weight': 10.0,
                         'penalty_weight': 1.0, 'truth': False,
                         'truncation_strategy': 'random', 'type': 'mmap'})
for d in eval_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}', 'sampling_weight': 0.0,
                         'penalty_weight': 1.0, 'truth': False,
                         'truncation_strategy': 'split', 'type': 'mmap'})

pos_feature = {'features_dir': 'generated_augmented_features/training',
               'sampling_weight': 2.0, 'penalty_weight': 1.0, 'truth': True,
               'truncation_strategy': 'truncate_start', 'type': 'mmap'}

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'spectrogram_length': 204,
    'stride': 3,
    'features': [pos_feature] + neg_features,
    'training_steps': [TRAINING_STEPS],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001],
    'batch_size': 128,
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': '',
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'\u2705 Config saved with {len(config["features"])} feature sources!')


## Step 12: Train Model

This takes 1-3 hours. Go get coffee!

In [ ]:
import subprocess, sys

print('Starting training...')
print(f'~{TRAINING_STEPS // 10000} hour(s)...')

train_cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train=1',
    # restore_checkpoint=0 for a fresh run — using 1 fails if no checkpoint exists yet
    '--restore_checkpoint', '0',
    '--test_tf_nonstreaming', '0',
    '--test_tflite_nonstreaming', '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming', '0',
    '--test_tflite_streaming_quantized', '1',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    # Removed the extra wrapping single-quotes — subprocess passes args directly,
    # shell quoting is not needed and was being passed as literal characters.
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]

# Capture stderr separately so errors are always visible at the bottom of the cell
# even if a lot of training log output scrolled by.
result = subprocess.run(train_cmd, stderr=subprocess.PIPE, text=True)

if result.returncode == 0:
    print('\u2705 Training complete!')
else:
    print('\u274c Training failed. Error output:')
    print(result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr)


## Step 13: Download Model

In [ ]:
import os
from google.colab import files

model_path = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'

if os.path.exists(model_path):
    print(f'\u2705 Model: {os.path.getsize(model_path)/1024:.1f} KB')
    files.download(model_path)
    print('\n\U0001f389 Done! Check your downloads.')
else:
    print('\u274c Model not found - check training output above')
